# Install & Import

In [1]:
!pip install -q sentence-transformers faiss-gpu-cu12 rank_bm25 transformers sacremoses 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 40.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 42.2 MB/s eta 0:00:00


In [2]:
import os
import re
import gc
import faiss
import pickle
import torch
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import MarianMTModel, MarianTokenizer

torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

# PATHS

In [3]:
DATA_DIR = "/kaggle/input/competitions/llm-agentic-legal-information-retrieval"
CACHE_DIR = "/kaggle/working/cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# LOAD DATA

In [4]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")

laws_de = pd.read_csv(f"{DATA_DIR}/laws_de.csv")
court_considerations = pd.read_csv(f"{DATA_DIR}/court_considerations.csv")

print("train:", train.shape)
print("test:", test.shape)
print("laws_de:", laws_de.shape)
print("court_considerations:", court_considerations.shape)

train: (1139, 3)
test: (40, 2)
laws_de: (175933, 3)
court_considerations: (2476315, 2)


In [5]:
laws_de.head()

,citation,text,title
0,Art. 1 112,Die Einwohnergemeinde Bern tritt der Schweizer...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
1,Art. 2 112,Die Einwohnergemeinde Bern wird ferner der Sch...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
2,Art. 3 Abs. 1 112,1 Falls die Schweizerische Eidgenossenschaft z...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
3,Art. 3 Abs. 2 112,2 Durch Anlage des neuen Verwaltungsgebäudes a...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
4,Art. 4 Abs. 1 112,1 Die Einwohnergemeinde Bern übernimmt im fern...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...


In [6]:
court_considerations.head()

,citation,text
0,BGE 139 I 2 E. 1.12.2011,betr. Verweigerung der Beiladung seien gutzuhe...
1,BGE 139 I 2 E. 2,Eventualiter sei die Rückweisung an die Vorins...
2,BGE 139 I 2 E. 5.1,"In der Sache ist vorweg zu prüfen, ob der Ents..."
3,BGE 139 I 2 E. 5.2,Art. 34 Abs. 1 BV gewährleistet in allgemeiner...
4,BGE 139 I 2 E. 5.3,Im vorliegenden Fall geht es nicht um die Gült...


laws_de is like law book

court_considerations is like case studies

| laws_de          | court_considerations |
| ---------------- | -------------------- |
| Rules            | Interpretations      |
| Short text       | Long text            |
| Structured       | Complex              |
| Easier retrieval | Harder retrieval     |


# Prepare Corpus

In [7]:
laws_de.columns

Index(['citation', 'text', 'title'], dtype='object')

In [8]:
#prepare corpus
def prepare(df):
    df["title"]=df.get("title","").fillna("")
    df["text"]=df["text"].fillna("")
    df["full_text"]=df["title"]*3+" "+df["text"]
    return df

laws_corpus=prepare(laws_de)

laws_docs = laws_corpus["full_text"].tolist()
laws_ids = laws_corpus["citation"].tolist()

court_docs=court_considerations["text"].tolist()
court_ids=court_considerations["citation"].tolist()

all_docs=laws_docs+court_docs
all_ids=laws_ids+court_ids


id2text =dict(zip(all_ids,all_docs))

del laws_docs, laws_ids, court_docs, court_ids, laws_corpus, laws_de, court_considerations

In [12]:
# !rm -rf /kaggle/working/cache

In [11]:
CACHE_DIR = "/kaggle/working/cache"
os.makedirs(CACHE_DIR, exist_ok=True)


np.save(CACHE_DIR+"/all_ids.npy", np.array(all_ids))

with open(CACHE_DIR+"/all_docs.pkl","wb") as f:
    pickle.dump(all_docs,f)

with open(CACHE_DIR+"/id2text.pkl","wb") as f:
    pickle.dump(id2text,f)

# Model Initialization

In [9]:
# cross Lingual
model_path= "/kaggle/input/models/yethukmutt/bge-m3/transformers/m3/1/bge-m3"
reranker_path="/kaggle/input/models/andreasbis/baai-bge-reranker-v2-m3/transformers/default/1"

embed_model =SentenceTransformer(model_path).to("cuda") # best choice
embed_model._first_module().auto_model.half()

reranker = CrossEncoder(reranker_path)
reranker.model.to("cuda")
reranker.model.half()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification(
  (classifier): XLMRobertaClassificationHead(
    (dense): Linear(in_features=1024, out_features=1024, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (out_proj): Linear(in_features=1024, out_features=1, bias=True)
  )
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(8194, 1024, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
             

In [12]:
embed_model.max_seq_length = 256   # unchanged from your setup

# Translation Batch Precompute

In [10]:
from transformers import MarianMTModel, MarianTokenizer, MarianConfig

translate_path="/kaggle/input/models/aaryanverma/helsinki-nlpopus-mt-en-de/transformers/default/1/opus-mt-en-de"
tokenizer=MarianTokenizer.from_pretrained(translate_path)
translator=MarianMTModel.from_pretrained(translate_path).to("cuda")
translator.eval()

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(58101, 512, padding_idx=58100)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(58101, 512, padding_idx=58100)
      (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): SiLU()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm): LayerNorm((512,), eps=1e-05

In [14]:
def translate_batch(texts, batch_size=32, max_length=256):
    outputs_all = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length
        ).to("cuda")

        with torch.no_grad():
            generated = translator.generate(**inputs, max_length=max_length)

        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        outputs_all.extend(decoded)

        del inputs, generated
        torch.cuda.empty_cache()
    return outputs_all

# Precompute translated test queries once
translated_test_path = f"{CACHE_DIR}/translated_test.pkl"

if os.path.exists(translated_test_path):
    with open(translated_test_path, "rb") as f:
        all_translated = pickle.load(f)
else:
    translated_queries = translate_batch(test["query"].tolist(), batch_size=32)
    all_translated = dict(zip(test["query"].tolist(), translated_queries))
    with open(translated_test_path, "wb") as f:
        pickle.dump(all_translated, f)

print("Translation cache ready.")

Translation cache ready.



# LOAD OR BUILD EMBEDDINGS


In [17]:
print(faiss.get_num_gpus())

2


In [18]:
# import gc, torch
# gc.collect()
# torch.cuda.empty_cache()

### CHUNKED EMBEDDING (WITH CHECKPOINTS)

In [19]:
import torch
import gc

torch.set_grad_enabled(False)

CHUNK_SIZE = 50000
emb_dir = CACHE_DIR + "/emb_chunks"
os.makedirs(emb_dir, exist_ok=True)

embed_model.max_seq_length = 256   # 🔥 CRITICAL FIX

num_chunks = (len(all_docs) + CHUNK_SIZE - 1) // CHUNK_SIZE

for i in range(num_chunks):
    chunk_path = f"{emb_dir}/chunk_{i}.npy"

    if os.path.exists(chunk_path):
        print(f"✅ Skipping chunk {i}")
        continue

    start = i * CHUNK_SIZE
    end = min((i + 1) * CHUNK_SIZE, len(all_docs))

    chunk_docs = all_docs[start:end]

    emb = embed_model.encode(
        chunk_docs,
        batch_size=16,          # 🔥 FIXED (was 128)
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True
    ).astype("float32")

    np.save(chunk_path, emb)

    # 🔥 memory cleanup
    del emb
    torch.cuda.empty_cache()
    gc.collect()

    print(f"💾 Saved chunk {i}")

✅ Skipping chunk 0
✅ Skipping chunk 1
✅ Skipping chunk 2
✅ Skipping chunk 3
✅ Skipping chunk 4
✅ Skipping chunk 5
✅ Skipping chunk 6
✅ Skipping chunk 7
✅ Skipping chunk 8
✅ Skipping chunk 9
✅ Skipping chunk 10
✅ Skipping chunk 11
✅ Skipping chunk 12
✅ Skipping chunk 13
✅ Skipping chunk 14
✅ Skipping chunk 15
✅ Skipping chunk 16
✅ Skipping chunk 17
✅ Skipping chunk 18
✅ Skipping chunk 19
✅ Skipping chunk 20
✅ Skipping chunk 21
✅ Skipping chunk 22
✅ Skipping chunk 23
✅ Skipping chunk 24
✅ Skipping chunk 25
✅ Skipping chunk 26
✅ Skipping chunk 27
✅ Skipping chunk 28
✅ Skipping chunk 29
✅ Skipping chunk 30
✅ Skipping chunk 31
✅ Skipping chunk 32
✅ Skipping chunk 33
✅ Skipping chunk 34
✅ Skipping chunk 35
✅ Skipping chunk 36
✅ Skipping chunk 37
✅ Skipping chunk 38
✅ Skipping chunk 39
✅ Skipping chunk 40
✅ Skipping chunk 41
✅ Skipping chunk 42
✅ Skipping chunk 43
✅ Skipping chunk 44
✅ Skipping chunk 45
✅ Skipping chunk 46
✅ Skipping chunk 47
✅ Skipping chunk 48
✅ Skipping chunk 49
✅ Skipping

# MERGE EMBEDDINGS

In [19]:
chunks = []
for i in range(num_chunks):
    chunk_path = f"{emb_dir}/chunk_{i}.npy"
    chunks.append(np.load(chunk_path))

embeddings = np.vstack(chunks)

np.save(CACHE_DIR + "/embeddings.npy", embeddings)

print("✅ All embeddings merged")

OSError: Not enough free space to write 10863607808 bytes

In [15]:
emb_path = f"{CACHE_DIR}/embeddings.npy"
index_path = f"{CACHE_DIR}/faiss.index"

if os.path.exists(emb_path):
    embeddings = np.load(emb_path)
    print("Loaded cached embeddings:", embeddings.shape)

Loaded cached embeddings: (2652248, 1024)


# 8. FAISS INDEX


In [ ]:
if os.path.exists(index_path):
    index = faiss.read_index(index_path)
    print("Loaded FAISS index.")
else:
    dim = embeddings.shape[1]
    quantizer = faiss.IndexFlatIP(dim)
    index = faiss.IndexIVFFlat(quantizer, dim, 2048, faiss.METRIC_INNER_PRODUCT)

    train_size = min(200000, len(embeddings))
    index.train(embeddings[:train_size])
    index.add(embeddings)
    index.nprobe = 60

    faiss.write_index(index, index_path)
    print("Built and saved FAISS index.")

index.nprobe = 60

# 9. BM25


In [ ]:
def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9äöüàáâãçéèêëìíîïñòóôõùúûüß§.\-/ ]", " ", text)
    return text.split()

bm25_path = f"{CACHE_DIR}/bm25.pkl"

if os.path.exists(bm25_path):
    with open(bm25_path, "rb") as f:
        bm25 = pickle.load(f)
    print("Loaded BM25.")
else:
    tokenized_docs = [tokenize(t) for t in tqdm(all_docs, desc="Tokenizing docs for BM25")]
    bm25 = BM25Okapi(tokenized_docs)
    with open(bm25_path, "wb") as f:
        pickle.dump(bm25, f)
    print("Built and saved BM25.")

# 10. RETRIEVAL HELPERS


In [17]:
def vector_search(query, k=80):
    q_emb = embed_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    scores, idxs = index.search(q_emb, k)
    results = []
    for idx, score in zip(idxs[0], scores[0]):
        if idx == -1:
            continue
        doc_id = all_ids[idx]
        results.append((doc_id, id2text[doc_id], float(score)))
    return results

def bm25_search(query, k=80):
    scores = bm25.get_scores(tokenize(query))
    idx = np.argsort(scores)[::-1][:k]
    return [(all_ids[i], all_docs[i], float(scores[i])) for i in idx]

def dedup_keep_best(results):
    best = {}
    for doc_id, text, score in results:
        if doc_id not in best or score > best[doc_id][1]:
            best[doc_id] = (text, score)
    return [(doc_id, best[doc_id][0], best[doc_id][1]) for doc_id in best]

def weighted_rrf(result_dict, weights=None, k=60):
    if weights is None:
        weights = {name: 1.0 for name in result_dict.keys()}

    scores = {}
    for name, results in result_dict.items():
        w = weights.get(name, 1.0)
        for rank, (doc_id, _, _) in enumerate(results):
            scores[doc_id] = scores.get(doc_id, 0.0) + w * (1.0 / (k + rank + 1))
    return scores

def pattern_boost(score_dict, query):
    q = query.lower()

    boosted = {}
    for doc_id, score in score_dict.items():
        bonus = 1.0

        if re.search(r"\bart\.?\s?\d+[a-z]?\b", q) and "Art." in doc_id:
            bonus *= 1.15
        if "bge" in q and "BGE" in doc_id:
            bonus *= 1.15
        if "§" in q and "§" in doc_id:
            bonus *= 1.10

        boosted[doc_id] = score * bonus
    return boosted

def select_candidate_docs(fused_scores, top_n=60):
    top_ids = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
    return [(doc_id, id2text[doc_id]) for doc_id, _ in top_ids]


# 11. RERANKING

In [ ]:
def rerank(query, docs, top_k=10):
    pairs = [[query, f"{doc_id} {text}"] for doc_id, text in docs]
    scores = reranker.predict(pairs, batch_size=32, show_progress_bar=False)

    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    reranked_ids = [doc_id for _, (doc_id, _) in ranked[:top_k]]
    return reranked_ids, ranked

# 12. DYNAMIC CITATION COUNT


In [ ]:
def choose_top_k_from_scores(ranked, min_k=3, max_k=12, margin_drop=0.08):
    scores = [float(s) for s, _ in ranked]
    if len(scores) == 0:
        return 0
    if len(scores) <= min_k:
        return len(scores)

    k = min_k
    for i in range(min_k, min(len(scores) - 1, max_k - 1)):
        drop = scores[i] - scores[i + 1]
        k = i + 1
        if drop >= margin_drop:
            break

    return min(k, max_k)

# 13. FULL PIPELINE

In [ ]:
def pipeline(query):
    q_en = query
    q_de = all_translated.get(query, query)

    vec_en = vector_search(q_en, k=80)
    vec_de = vector_search(q_de, k=80)
    bm_en = bm25_search(q_en, k=80)
    bm_de = bm25_search(q_de, k=80)

    vec = dedup_keep_best(vec_en + vec_de)
    bm = dedup_keep_best(bm_en + bm_de)

    fused = weighted_rrf(
        {
            "vector": vec,
            "bm25": bm
        },
        weights={
            "vector": 1.15,
            "bm25": 1.0
        }
    )

    fused = pattern_boost(fused, query)
    docs = select_candidate_docs(fused, top_n=60)

    top_ids_12, ranked = rerank(query, docs, top_k=12)
    chosen_k = choose_top_k_from_scores(ranked, min_k=3, max_k=12, margin_drop=0.08)

    final_ids = top_ids_12[:chosen_k]

    # exact corpus citation strings only
    final_ids = list(dict.fromkeys(final_ids))
    return ";".join(final_ids)

# 14. INFERENCE

In [ ]:
preds = []
for q in tqdm(test["query"].tolist(), desc="Running inference"):
    preds.append(pipeline(q))

submission = pd.DataFrame({
    "query_id": test["query_id"],
    "predicted_citations": preds
})

submission.to_csv("submission.csv", index=False)
print(submission.head())
print("Saved submission.csv")

# 15. CLEANUP

In [ ]:
gc.collect()
torch.cuda.empty_cache()